# 03 — Certified random MPUs, Mode II (fusion closure) and named families

In Mode II the tensor is diagonal on the bond, $A[a,b]=\delta_{ab}\,y_a$, so the chain collapses to

$$U_N=\sum_u \ell_u\, y_u^{\otimes N}.$$

Unitarity for every $N$ follows from **fusion closure** $y_u^\dagger y_v\in\{0\}\cup\{y_w\}$ together with a boundary on the skeleton's *admissible manifold*. That manifold is in general not the naive sphere $\sum|\ell_u|^2=1$ (paper, `thm:main1`).

In [1]:
import numpy as np

from mpu_generators import FAMILIES, UniformMPUGenerator, build_family
from TreeMPU import dense_unitary_from_tensors

## The collapse formula, checked explicitly

The channels are recovered from the diagonal blocks, $y_u$ = `A[:, :, u, u].T`, and $\sum_u\ell_u y_u^{\otimes N}$ is compared with the full contraction:

In [2]:
t, l, r = UniformMPUGenerator(seed=2).fusion(d=3, bond_dim=3, n_sites=3, skeleton="nested_flag")
y = [t[0][:, :, u, u].T for u in range(3)]
N = 3
U_collapse = sum(l[u] * np.kron(np.kron(y[u], y[u]), y[u]) for u in range(3))
U_full = dense_unitary_from_tensors(t, l, r)
print("collapse formula == contraction:", np.allclose(U_collapse, U_full))
print("unitary:", np.allclose(U_full.conj().T @ U_full, np.eye(27)))
print(
    "fusion closure y_u^dag y_v = y_max(u,v):",
    all(np.allclose(y[u].conj().T @ y[v], y[max(u, v)]) for u in range(3) for v in range(3)),
)

collapse formula == contraction: True
unitary: True
fusion closure y_u^dag y_v = y_max(u,v): True


## The skeletons

| `skeleton` | channels | closure | admissible boundary |
|---|---|---|---|
| `"group"` | left-regular representation of a finite group, $d=\chi=|G|$ | $y_g^\dagger y_h=y_{g^{-1}h}$ | Peter–Weyl manifold $\prod_\rho U(d_\rho)$ |
| `"cyclic"` | $\mathbb Z_M$ characters | $y_u^\dagger y_v=y_{v-u}$ | flat Fourier spectrum (a group convolution only if $M\mid d$) |
| `"nested_flag"` | nested projectors, contains MCZ | $y_u^\dagger y_v=y_{\max(u,v)}$ | flat torus $\ell_w=-S_w+e^{i\theta_w}$ |
| `"matrix_units"` | $\mathbb 1$ and $w_iw_j^\dagger$, contains subspace rotations | matrix-unit algebra | $\ell_{(i,j)}=(V-\mathbb 1)_{ij}$, $V\in U(m)$ |

Group convolution also covers non-abelian groups, for example $S_3$:

In [3]:
t, l, r = UniformMPUGenerator(seed=0).fusion(
    d=6, bond_dim=6, n_sites=2, skeleton="group", group="S3"
)
U = dense_unitary_from_tensors(t, l, r)
print("S3 group convolution, N=2: unitary =", np.allclose(U.conj().T @ U, np.eye(36)))
print(
    "boundary |l|^2 sums to",
    round(float(np.sum(np.abs(l) ** 2)), 12),
    "(necessary, not sufficient)",
)

S3 group convolution, N=2: unitary = True
boundary |l|^2 sums to 1.0 (necessary, not sufficient)


## Named families

`build_family` builds the families used in the thesis and the paper with the exact production settings. `FAMILIES` lists them:

In [4]:
for name, description in FAMILIES.items():
    print(f"{name:32s} {description}")

branching_uniform                Mode I, translation invariant (UniformMPUGenerator.branching)
branching_general                Mode I, site dependent (MPUGenerator.branching)
grafted                          Grafted composition (UniformMPUGenerator.grafted)
fusion_cyclic                    Mode II, cyclic skeleton (requires bond_dim <= d)
fusion_nested_flag               Mode II, nested flags (requires bond_dim <= d)
fusion_matrix_units              Mode II, matrix units (bond_dim = 1 + m^2, m <= d)
fusion_group_Zn                  Mode II, group convolution Z_n (requires d = bond_dim = n)
fusion_group_S3                  Mode II, group convolution S_3 (d = bond_dim = 6)
fusion_group_Zn_product          Mode II, group convolution Z_2 x Z_2 x Z_2 (d = bond_dim = 8)
fusion_bond_resolved_lee_yang    Mode II-B, Lee-Yang skeleton (d = 5, bond_dim = 14)


In [5]:
for name in [
    "branching_uniform",
    "fusion_cyclic",
    "fusion_nested_flag",
    "fusion_group_Zn",
    "grafted",
]:
    t, l, r = build_family(name, d=2, bond_dim=2, seed=1, n_sites=4)
    U = dense_unitary_from_tensors(t, l, r)
    print(f"{name:20s} chi={len(l)}  unitary={np.allclose(U.conj().T @ U, np.eye(U.shape[0]))}")

branching_uniform    chi=2  unitary=True
fusion_cyclic        chi=2  unitary=True
fusion_nested_flag   chi=2  unitary=True
fusion_group_Zn      chi=2  unitary=True
grafted              chi=2  unitary=True


## Combining certified chains

`mixing.stack` composes two chains, $U^{(1)}_NU^{(2)}_N$. `mixing.capped_sum` prepends one readout site carrying a row-resolution pair, giving $\Pi_1\otimes U^{(1)}_N+\Pi_2\otimes U^{(2)}_N$. Both are unitary for any unitary inputs; the naive sum of boundaries is not.

In [6]:
from mpu_generators.mixing import capped_sum, stack

m1 = build_family("branching_uniform", d=2, bond_dim=2, seed=0, n_sites=3)
m2 = build_family("fusion_cyclic", d=2, bond_dim=2, seed=1, n_sites=3)
for name, (t, l, r) in [("stack", stack(m1, m2)), ("capped_sum", capped_sum(m1, m2, seed=0))]:
    U = dense_unitary_from_tensors(t, l, r)
    print(
        f"{name:10s} tensors={len(t)} chi={len(l) if name == 'stack' else t[1].shape[2]}  unitary={np.allclose(U.conj().T @ U, np.eye(U.shape[0]))}"
    )

stack      tensors=3 chi=4  unitary=True
capped_sum tensors=4 chi=4  unitary=True


## Mode II-B: bond-resolved fusion (Lee–Yang)

Non-monomial fusion rules such as $\sigma\cdot\sigma=e+\sigma$ cannot be realized by tensor-power channels. In Mode II-B each channel is itself an MPO, and the rule is certified by an **injective intertwiner** $V$ with $B_{\sigma\sigma}V=V(B_e\oplus B_\sigma)$ (thesis `eq:intertwiner`). For Lee–Yang, $V$ maps $\mathbb C^{2+3}\to\mathbb C^{9}$.

Unlike Modes I and II, the intertwiner is the only algebraic check here: unitarity of the whole chain is verified numerically (dense for $d^N\le2^{10}$, normalization beyond).

In [7]:
from mpu_generators.skeletons import (
    lee_yang_skeleton,
    mpo_product_tensor,
    solve_intertwiner,
    verify_intertwiner,
)

sk = lee_yang_skeleton()
B_ss = mpo_product_tensor(sk.channels["sigma"], sk.channels["sigma"])
V = solve_intertwiner(B_ss, [sk.channels["e"], sk.channels["sigma"]])
print(
    "V shape:",
    V.shape,
    "| rank:",
    np.linalg.matrix_rank(V),
    "| residual of (I):",
    f"{verify_intertwiner(B_ss, [sk.channels['e'], sk.channels['sigma']], V):.1e}",
)

t, l, r = UniformMPUGenerator(seed=0).fusion_bond_resolved(
    lee_yang_skeleton(alpha=0.3, beta=1.7), n_sites=2
)
U = dense_unitary_from_tensors(t, l, r)
print("Lee-Yang (alpha=0.3, beta=1.7), N=2: unitary =", np.allclose(U.conj().T @ U, np.eye(25)))

V shape: (9, 5) | rank: 5 | residual of (I): 1.1e-16
Lee-Yang (alpha=0.3, beta=1.7), N=2: unitary = True


**Next:** notebook 04 turns these MPUs into quantum circuits with `TreeMPU`.